# Day 1 — Hands-on 5: Pretrained Models with Hugging Face Transformers

**Goal:** Run a pretrained model for sentiment analysis, inspect its confidence score, and distinguish inference from fine-tuning.

**Environment:** Google Colab recommended. This notebook is separate from the course LangChain environment because it installs PyTorch and Transformers. Internet access is needed for the first package/model download.

## Learning outcomes
By the end, you should be able to: load a pretrained model, run predictions, interpret labels and scores, test multiple examples, and explain why inference does not update model weights.

In [1]:
!pip -q install "transformers>=4.40,<5" torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import torch
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if DEVICE == 0 else "CPU")
MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
sentiment = pipeline("sentiment-analysis", model=MODEL_NAME, device=DEVICE)
print("Loaded:", MODEL_NAME)

Device: CPU


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


Loaded: distilbert/distilbert-base-uncased-finetuned-sst-2-english


## 1. Run your first predictions

In [3]:
reviews = [
    "The staff were helpful and the product arrived early.",
    "The app crashes every time I open it. Very disappointing.",
    "The product is okay, but the price is too high."
]
results = sentiment(reviews)
for text, result in zip(reviews, results):
    print(f"Review: {text}\nPrediction: {result['label']} | score: {result['score']:.3f}\n")

Review: The staff were helpful and the product arrived early.
Prediction: POSITIVE | score: 0.983

Review: The app crashes every time I open it. Very disappointing.
Prediction: NEGATIVE | score: 1.000

Review: The product is okay, but the price is too high.
Prediction: NEGATIVE | score: 0.996



**Interpretation:** `POSITIVE` / `NEGATIVE` is the predicted class. `score` is the model score for that predicted class; it is not a guarantee of correctness. This particular checkpoint is English-focused.

## 2. Test your own examples

In [4]:
my_reviews = [
    "Replace this with your own review.",
    "The delivery was late, but customer service solved the problem.",
    "I expected more for the money I paid."
]
for text, result in zip(my_reviews, sentiment(my_reviews)):
    print(f"{result['label']:8s} {result['score']:.3f} | {text}")

NEGATIVE 0.998 | Replace this with your own review.
NEGATIVE 0.990 | The delivery was late, but customer service solved the problem.
NEGATIVE 0.999 | I expected more for the money I paid.


## 3. Inspect model output and limitations

In [5]:
example = sentiment("The food was cold, but the staff were kind.")[0]
print(example)
print("Output keys:", list(example.keys()))

{'label': 'POSITIVE', 'score': 0.9977248311042786}
Output keys: ['label', 'score']


A single overall sentiment label can hide mixed opinions. The model may miss sarcasm, context, domain-specific meanings, and languages it was not trained for. Do not use this toy exercise to make high-stakes decisions about people.

## 4. Inference versus fine-tuning
- **Inference:** use the learned weights to make a prediction; no training/weight update happens in the pipeline calls above.
- **Fine-tuning:** train a pretrained model further on task-specific labelled examples; the training process updates weights. The next notebook demonstrates this.

## Student task
1. Write five positive, negative, or mixed reviews.
2. Record predicted label and score for each.
3. Identify one prediction that may be misleading and explain why.
4. In two sentences, distinguish inference from fine-tuning.

**Submission:** code output or screenshots plus a short observation table.

## Troubleshooting
- If a download fails, check internet access and rerun the model-loading cell.
- On CPU, loading/inference may be slower; that is normal.
- If Colab runtime resets, rerun cells from the top.
- This notebook uses a pretrained model; it does not train or update its weights.